# Backend Verification for the wafer.space GF180MCU Run-1

[D. Mitch Bailey](https://www.linkedin.com/in/mitch-bailey-cvc/), [ShuhariSystem](https://www.shuharisystem.com/)

H. Ojiro, [ShuhariSystem](https://www.shuharisystem.com/)

# Overview
This notebook performs backend LVS (with [klayout](https://github.com/KLayout/klayout) or [magic](https://github.com/RTimothyEdwards/magic)/[netgen](https://github.com/RTimothyEdwards/netgen)), soft-connection checks (with magic/netgen) and/or reliability verfication (with [CVC-RV](https://github.com/d-m-bailey/cvc)) with open source EDA software and systems.

# wafer.space GF180MCU Run 1
The 38 submissions to wafer.space shuttle for ws-run1 listed below are located [here](https://github.com/wafer-space/ws-run1).

| No | Code | Chip Top |
| -- | ---- | -------- |
| 1 | 2975 | 2975_chip_top_0_6 |
| 2 | 2975 | 2975_chip_top_2_6 |
| 3 | AS03 | AS03_chip_top_6_0 |
| 4 | BRWN | BRWN_ENGN2912E_TOP_8_2 |
| 5 | BTAP | BTAP_chip_top_0_0 |
| 6 | BTAP | BTAP_chip_top_2_0 |
| 7 | CAFE | CAFE_chip_top_12_0 |
| 8 | CHES | CHES_chip_top_10_4 |
| 9 | GD02 | GD02_chip_top_14_4 |
| 10| GD03 | GD03_chip_top_6_6 |
| 11 | GD04 | GD04_chip_top_10_8 |
| 12 | GD04 | GD04_chip_top_6_8 |
| 13 | HZ80 | HZ80_chip_top_14_0 |
| 14 | ISHI | ISHI_ISHI-KAI_WS_RUN1_12_4 |
| 15 | JKU1 | JKU1_chip_top_10_0 |
| 16 | JKU2 | JKU2_chip_top_0_8 |
| 17 | KIAN | KIAN_chip_top_8_0 |
| 18 | MOLE | MOLE_chip_top_8_4 |
| 19 | MOS2 | MOS2_chip_top_10_6 |
| 20 | MOSB | MOSB_chip_top_4_0 |
| 21 | MOSB | MOSB_chip_top_4_2 |
| 22 | OCD1 | OCD1_caravel_openframe_top_8_6 |
| 23 | OCD2 | OCD2_gf180mcu_ocd_sram_top_2_8 |
| 24 | OCD2 | OCD2_gf180mcu_ocd_sram_top_4_8 |
| 25 | RBOY | RBOY_chip_top_12_6 |
| 26 | RZ80 | RZ80_chip_top_0_2 |
| 27 | RZ80 | RZ80_chip_top_2_2 |
| 28 | RZML | RZML_chip_top_12_2 |
| 29 | TQVA | TQVA_chip_top_14_8 |
| 30 | TQVB | TQVB_chip_top_14_6 |
| 31 | TQVC | TQVC_chip_top_12_8 |
| 32 | TQVC | TQVC_chip_top_8_8 |
| 33 | TRID | TRID_TOP_14_2 |
| 34 | TTP2 | TTP2_tt_gf_wrapper_6_4 |
| 35 | TTPG | TTPG_tt_gf_wrapper_6_2 |
| 36 | TZ01 | TZ01_chip_top_0_4 |
| 37 | TZ01 | TZ01_chip_top_2_4 |
| 38 | WSLG | WSLG_chip_top_10_2 |

## I. Set initial environment variables.

In [17]:
%%writefile /content/env
export LOCAL_INSTALL=/content/local
export PATH=$PATH:$LOCAL_INSTALL/bin
export PDK_ROOT=/content/pdks
export PDK=gf180mcuD
export PDKPATH=$PDK_ROOT/$PDK
export PDK_COMMIT=f6eeac7dad085ffcc829ccfd721f7b4ce39edcf7
export MAGIC_VERSION=8.3.693
export NETGEN_VERSION=1.5.323
export KLAYOUT_DISTRO=Ubuntu-24
export KLAYOUT_DEB=klayout_0.30.12-1_amd64.deb
export EXTRA_CHECK_VERSION=wafer.space-run1
export CVC_VERSION=master
export DATA_ROOT=/content/design_data
export CHECK_VERSION=main
export LVS_ROOT=/root/extra_be_checks
export MPW_ROOT=/content/ws-mpw-be
export MPW=ws-run1
export SHUTTLE_ID=G801
if [[ -f $DATA_ROOT/project_env ]]; then
  cat $DATA_ROOT/project_env
  source $DATA_ROOT/project_env
fi

Overwriting /content/env


# II. Program installation.
Only needs to be executed once.

This step sets up the pdk and installs magic, klayout, netgen and cvc_rv.

Runtime: 3-4 minutes

Note: klayout LVS rule files are modified as follows:



Runtime: ~ 5 minutes.

In [18]:
%%shell
cd
cat /content/env
source /content/env

lsb_release -a 2>/dev/null

sudo apt update

if ! command -v ciel; then
  echo "==> Installing ciel..."
  pip install ciel
fi
rm -rf ${PDK_ROOT}
ciel enable --pdk $PDK $PDK_COMMIT
echo "==> Using pdk $PDK commit $PDK_COMMIT (patched)"

echo "==> Downloading extra_be_checks $EXTRA_CHECK_VERSION"
rm -rf extra_be_checks
# git clone https://github.com/d-m-bailey/extra_be_checks.git -b $EXTRA_CHECK_VERSION
git clone https://github.com/H-Ojiro/extra_be_checks.git -b $EXTRA_CHECK_VERSION
echo "==> Using extra_be_checks commit $(cd extra_be_checks; git rev-parse HEAD) "

if ! command -v netgen; then
  echo "==> Downloading and installing netgen $NETGEN_VERSION"
  git clone https://github.com/RTimothyEdwards/netgen.git --depth=1 -b $NETGEN_VERSION
  cd netgen
  ./configure --prefix=$LOCAL_INSTALL
  make
  make install
  cd
fi
echo "==> Using netgen version $(netgen -batch | awk '/Netgen/ {print $2}')"

if ! command -v magic; then
  echo "==> Downloading and installing magic $MAGIC_VERSION"
  git clone https://github.com/RTimothyEdwards/magic.git --depth=1 -b $MAGIC_VERSION
  cd magic
  ./configure --prefix=$LOCAL_INSTALL
  make
  make install
  cd
fi
echo "==> Using magic version $(magic -dnull -noc --version)"

if ! command -v klayout; then
  echo "==> Downloading and installing klayout $KLAYOUT_DEB for $KLAYOUT_DISTRO"
  wget -P /tmp https://www.klayout.org/downloads/$KLAYOUT_DISTRO/$KLAYOUT_DEB
  sudo dpkg -i /tmp/$KLAYOUT_DEB
  sudo apt --fix-broken install
  pip install docopt
  pip install klayout
  cd
fi
echo "==> Using $(klayout -v)"

if ! command -v cvc_rv; then
  echo "==> Downloading and installing cvc_rv $CVC_VERSION"
  sudo apt install autopoint bison flex
  git clone https://github.com/d-m-bailey/cvc --depth=1 -b $CVC_VERSION
  cd cvc
  autoreconf -vif
  ./configure --prefix=$LOCAL_INSTALL --disable-nls
  make
  make install
  cd
fi
echo "==> Using $(cvc_rv -v)"

echo "#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~"
echo "pdk $PDK commit $PDK_COMMIT (patched)"
echo "extra_be_checks commit $(cd extra_be_checks; git rev-parse HEAD)"
echo "netgen version $(netgen -batch | awk '/Netgen/ {print $2}')"
echo "magic version $(magic -dnull -noc --version)"
echo "$(klayout -v)"
echo "$(cvc_rv -v)"
echo "#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~"

export LOCAL_INSTALL=/content/local
export PATH=$PATH:$LOCAL_INSTALL/bin
export PDK_ROOT=/content/pdks
export PDK=gf180mcuD
export PDKPATH=$PDK_ROOT/$PDK
export PDK_COMMIT=f6eeac7dad085ffcc829ccfd721f7b4ce39edcf7
export MAGIC_VERSION=8.3.693
export NETGEN_VERSION=1.5.323
export KLAYOUT_DISTRO=Ubuntu-24
export KLAYOUT_DEB=klayout_0.30.12-1_amd64.deb
export EXTRA_CHECK_VERSION=wafer.space-run1
export CVC_VERSION=master
export DATA_ROOT=/content/design_data
export CHECK_VERSION=main
export LVS_ROOT=/root/extra_be_checks
export MPW_ROOT=/content/ws-mpw-be
export MPW=ws-run1
export SHUTTLE_ID=G801
if [[ -f $DATA_ROOT/project_env ]]; then
  cat $DATA_ROOT/project_env
  source $DATA_ROOT/project_env
fi
export LAYOUT_TOP=TQVA_chip_top_14_8
# export LAYOUT_TOP=2975_chip_top_0_6
export PROJECT=${LAYOUT_TOP}
export CHIP_ID=${LAYOUT_TOP%%_*}
export UPRJ_ROOT=${DATA_ROOT}/${CHIP_ID}
export WORK_ROOT=${DATA_ROOT}/${CHIP_ID}/work
export CHIP_ROOT=${MPW_ROOT}/layout/${CHIP_ID}
export LAYOUT_DATA=${CHI

# III. Create the design repo and extract the GDS file for each design.

Run to refresh the design data.

Runtime: ~10 minutes.

In [19]:
%%shell
cat /content/env
source /content/env
# rm -rfv ${DATA_ROOT}
# rm -rfv ${MPW_ROOT}
if [[ ! -d ${DATA_ROOT} ]]; then
  git clone https://github.com/wafer-space/$MPW.git --depth=1 ${DATA_ROOT}
fi
if [[ ! -d ${MPW_ROOT} ]]; then
  git clone https://github.com/H-Ojiro/ws-mpw-be.git --depth=1 -b ${CHECK_VERSION} ${MPW_ROOT}
fi

# Assemble G801.oas from the individual files
pushd ${DATA_ROOT}/layout
if [[ ! -e ${SHUTTLE_ID}.oas ]]; then
  echo "Merging ${SHUTTLE_ID}.oas .."
  ./create-reticle-oas.sh
fi
ls -l *.oas
popd

# Extract gds file of each mpw chip
pushd ${MPW_ROOT}/layout
echo "Creating a chip gds file .."
${MPW_ROOT}/scripts/create_chip_gds.sh ${DATA_ROOT}/layout/${SHUTTLE_ID}.oas


export LOCAL_INSTALL=/content/local
export PATH=$PATH:$LOCAL_INSTALL/bin
export PDK_ROOT=/content/pdks
export PDK=gf180mcuD
export PDKPATH=$PDK_ROOT/$PDK
export PDK_COMMIT=f6eeac7dad085ffcc829ccfd721f7b4ce39edcf7
export MAGIC_VERSION=8.3.693
export NETGEN_VERSION=1.5.323
export KLAYOUT_DISTRO=Ubuntu-24
export KLAYOUT_DEB=klayout_0.30.12-1_amd64.deb
export EXTRA_CHECK_VERSION=wafer.space-run1
export CVC_VERSION=master
export DATA_ROOT=/content/design_data
export CHECK_VERSION=main
export LVS_ROOT=/root/extra_be_checks
export MPW_ROOT=/content/ws-mpw-be
export MPW=ws-run1
export SHUTTLE_ID=G801
if [[ -f $DATA_ROOT/project_env ]]; then
  cat $DATA_ROOT/project_env
  source $DATA_ROOT/project_env
fi
export LAYOUT_TOP=TQVA_chip_top_14_8
# export LAYOUT_TOP=2975_chip_top_0_6
export PROJECT=${LAYOUT_TOP}
export CHIP_ID=${LAYOUT_TOP%%_*}
export UPRJ_ROOT=${DATA_ROOT}/${CHIP_ID}
export WORK_ROOT=${DATA_ROOT}/${CHIP_ID}/work
export CHIP_ROOT=${MPW_ROOT}/layout/${CHIP_ID}
export LAYOUT_DATA=${CHI

# ※ Run cells before this cell for program installation and general setup.<br>Then choose one setup from the 38 designs below.<br>Finally, run the checks and examine the results

General results:
1. When the layout has cells that have the same name as primitive devices, netgen segfaults. <br>➡︎ Flatglob all layout cells with primitive device names.

# IV. Setup Chip Project
### Check the chip GDS on Klayout GUI
- Some chips' GDS might be missing top level pad texts.
- Copy up corresponding `Metal5_Label 81/10`
pad texts from the hierarchical block to top level.
- Check pin.list in design_data/\<CHIP_ID\>/work directory.

## 1. Setup project, top cell and gds file names.

In [20]:
%%shell
source /content/env

if [[ ! -e ${MPW_ROOT}/layout/chip_list.txt ]]; then
  echo "Can't find chip_list.txt"
  exit 1
fi

mapfile -t chip_list < "${MPW_ROOT}/layout/chip_list.txt"

if [[ -z ${chip_list} ]]; then
  echo "$0: Undefined chip list"
  exit 1
fi

chip_list+=('Cancel')

PS3='Select number> '
trap "echo Exiting. Please input a number again; exit" SIGINT
select chip_name in "${chip_list[@]}"; do
  [[ -z "${REPLY}" ]] && {
    echo "Please enter a number."
    break
  }
  [[ -z "${chip_name}" ]] && {
    echo "##### ERROR: Invalid selection = ${REPLY}"
    break
  }
  if [[ "${chip_name}" == Cancel ]]; then
    echo "-> Job canceled"
    exit
  fi

  break 2
done

echo "export LAYOUT_TOP=${chip_name}"

{
  echo "export LAYOUT_TOP=${chip_name}"

  cat <<'EOF'
# export LAYOUT_TOP=2975_chip_top_0_6
export PROJECT=${LAYOUT_TOP}
export CHIP_ID=${LAYOUT_TOP%%_*}
export UPRJ_ROOT=${DATA_ROOT}/${CHIP_ID}
export WORK_ROOT=${DATA_ROOT}/${CHIP_ID}/work
export CHIP_ROOT=${MPW_ROOT}/layout/${CHIP_ID}
export LAYOUT_DATA=${CHIP_ROOT}/${LAYOUT_TOP}.gds.gz
export CELL_LIST=${CHIP_ROOT}/cell.list
export PIN_LIST=${CHIP_ROOT}/pin.list
export SC_SPICE_FILES=${PDK_ROOT}/${PDK}/libs.ref/gf180mcu_fd_sc_mcu9t5v0/spice/*.spice
export SRAM_SPICE_FILES=${PDK_ROOT}/${PDK}/libs.ref/gf180mcu_fd_ip_sram/spice/*.spice
export CDL_FILE=
export CARAVEL_ROOT=
export SRAM_MACRO=gf180mcu_fd_ip_sram__sram512x8m8wm1
EOF
} > "${DATA_ROOT}/project_env"

cat ${DATA_ROOT}/project_env
source ${DATA_ROOT}/project_env

if [[ ! -e ${LAYOUT_DATA} ]]; then
  echo "Can't find ${LAYOUT_DATA}"
  exit 1
fi

if [[ -e ${CHIP_ROOT}/add_pad_text.json ]]; then
  if [[ ! -e ${LAYOUT_DATA}.bak ]]; then
    mv -fv ${LAYOUT_DATA} ${LAYOUT_DATA}.bak
    echo "Running ${MPW_ROOT}/scripts/apply_json_to_layout.py .."
    ${MPW_ROOT}/scripts/apply_json_to_layout.py \
      -i ${LAYOUT_DATA}.bak \
      -j ${CHIP_ROOT}/add_pad_text.json \
      -o ${LAYOUT_DATA}
  fi
fi

rm -f ${DATA_ROOT}/work
rm -rf ${WORK_ROOT}
mkdir -p ${WORK_ROOT}
ln -s ${WORK_ROOT} ${DATA_ROOT}/work
cd ${DATA_ROOT}/work
ln -s ${LAYOUT_DATA} .
ln -s ${CELL_LIST} .
ln -s ${PIN_LIST} .
ls -l


export LAYOUT_TOP=TQVA_chip_top_14_8
# export LAYOUT_TOP=2975_chip_top_0_6
export PROJECT=${LAYOUT_TOP}
export CHIP_ID=${LAYOUT_TOP%%_*}
export UPRJ_ROOT=${DATA_ROOT}/${CHIP_ID}
export WORK_ROOT=${DATA_ROOT}/${CHIP_ID}/work
export CHIP_ROOT=${MPW_ROOT}/layout/${CHIP_ID}
export LAYOUT_DATA=${CHIP_ROOT}/${LAYOUT_TOP}.gds.gz
export CELL_LIST=${CHIP_ROOT}/cell.list
export PIN_LIST=${CHIP_ROOT}/pin.list
export SC_SPICE_FILES=${PDK_ROOT}/${PDK}/libs.ref/gf180mcu_fd_sc_mcu9t5v0/spice/*.spice
export SRAM_SPICE_FILES=${PDK_ROOT}/${PDK}/libs.ref/gf180mcu_fd_ip_sram/spice/*.spice
export CDL_FILE=
export CARAVEL_ROOT=
export SRAM_MACRO=gf180mcu_fd_ip_sram__sram512x8m8wm1
 1) 2975_chip_top_0_6		    21) MOSB_chip_top_4_2
 2) 2975_chip_top_2_6		    22) OCD1_caravel_openframe_top_8_6
 3) AS03_chip_top_6_0		    23) OCD2_gf180mcu_ocd_sram_top_2_8
 4) BRWN_ENGN2912E_TOP_8_2	    24) OCD2_gf180mcu_ocd_sram_top_4_8
 5) BTAP_chip_top_0_0		    25) RBOY_chip_top_12_6
 6) BTAP_chip_top_2_0		    26) RZ80_chip_to

## 2. Create `lvs_config.json` and update the `LVS_SPICE_FILES` and `LVS_VERILOG_FILES` for every design.

- Update the other parameters as needed.

In [25]:
%%shell
cat /content/env
source /content/env

# Remove STD_CELL fill cells from the abstract
sed -i.bak '/__fill_/d; /__endcap/d; /__filltie/ s/^.*$/\t\t""/' ${LVS_ROOT}/tech/${PDK}/lvs_config.base.json

if (( $(grep -c 'gf180mcu_ocd_ip_sram__sram' ${CELL_LIST}) > 0 )); then
  sed -e 's/\$SRAM_MACRO/gf180mcu_ocd_ip_sram__sram64x8m8wm1/' ${LVS_ROOT}/tech/${PDK}/lvs_config.sram.json \
    > ${WORK_ROOT}/lvs_config.sram64.json
  sed -e 's/\$SRAM_MACRO/gf180mcu_ocd_ip_sram__sram128x8m8wm1/' ${LVS_ROOT}/tech/${PDK}/lvs_config.sram.json \
    > ${WORK_ROOT}/lvs_config.sram128.json
  sed -e 's/\$SRAM_MACRO/gf180mcu_ocd_ip_sram__sram256x8m8wm1/' ${LVS_ROOT}/tech/${PDK}/lvs_config.sram.json \
    > ${WORK_ROOT}/lvs_config.sram256.json
  sed -e 's/\$SRAM_MACRO/gf180mcu_ocd_ip_sram__sram512x8m8wm1/' ${LVS_ROOT}/tech/${PDK}/lvs_config.sram.json \
    > ${WORK_ROOT}/lvs_config.sram512.json
else
  sed -e 's/\$SRAM_MACRO/gf180mcu_fd_ip_sram__sram64x8m8wm1/' ${LVS_ROOT}/tech/${PDK}/lvs_config.sram.json \
    > ${WORK_ROOT}/lvs_config.sram64.json
  sed -e 's/\$SRAM_MACRO/gf180mcu_fd_ip_sram__sram128x8m8wm1/' ${LVS_ROOT}/tech/${PDK}/lvs_config.sram.json \
    > ${WORK_ROOT}/lvs_config.sram128.json
  sed -e 's/\$SRAM_MACRO/gf180mcu_fd_ip_sram__sram256x8m8wm1/' ${LVS_ROOT}/tech/${PDK}/lvs_config.sram.json \
    > ${WORK_ROOT}/lvs_config.sram256.json
  sed -e 's/\$SRAM_MACRO/gf180mcu_fd_ip_sram__sram512x8m8wm1/' ${LVS_ROOT}/tech/${PDK}/lvs_config.sram.json \
    > ${WORK_ROOT}/lvs_config.sram512.json
fi

if (( $(grep -c 'gf180mcu_ocd_io__' ${CELL_LIST}) > 0 )); then
  sed -e 's/gf180mcu_fd_io/gf180mcu_ocd_io/' ${LVS_ROOT}/tech/${PDK}/lvs_config.chip_io.json \
    > ${WORK_ROOT}/lvs_config.chip_io.json
else
  cp -pf ${LVS_ROOT}/tech/${PDK}/lvs_config.chip_io.json \
    ${WORK_ROOT}/lvs_config.chip_io.json
fi

cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "STD_CELL_LIBRARY": "gf180mcu_fd_sc_mcu7t5v0",
  "INCLUDE_CONFIGS": [
     "$WORK_ROOT/lvs_config.sram64.json",
     "$WORK_ROOT/lvs_config.sram128.json",
     "$WORK_ROOT/lvs_config.sram256.json",
     "$WORK_ROOT/lvs_config.sram512.json",
     "$WORK_ROOT/lvs_config.chip_io.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "VIA_*",
    "GR_NMOS*",
    "*_route*",
    "*_via*",
    "via*",
    "TILE_*",
    "np*",
    "pn*",
    "diode_nd2ps*",
    "diode_pd2nw*",
    "condiode*",
    "ppolyf_*",
    "short*",
    "efuse*",
    "nfet*",
    "pfet*",
    "nmos*",
    "pmos*",
    "mim*",
    "cap_mim*",
    "cap_nmos*",
    "pnp_*",
    "lv_*",
    "tie_poly_res*",
    "lvlshift_*",
    "sealring*"
  ],
  "EXTRACT_ABSTRACT": [
    "gf180mcu_fd_io__cor*",
    "gf180mcu_ocd_io__cor*",
    "gf180mcu_fd_ip_sram*",
    "gf180mcu_ocd_ip_sram*"
  ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SC_SPICE_FILES",
    "$SRAM_SPICE_FILES"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

cat ${WORK_ROOT}/lvs_config.json

export LOCAL_INSTALL=/content/local
export PATH=$PATH:$LOCAL_INSTALL/bin
export PDK_ROOT=/content/pdks
export PDK=gf180mcuD
export PDKPATH=$PDK_ROOT/$PDK
export PDK_COMMIT=f6eeac7dad085ffcc829ccfd721f7b4ce39edcf7
export MAGIC_VERSION=8.3.693
export NETGEN_VERSION=1.5.323
export KLAYOUT_DISTRO=Ubuntu-24
export KLAYOUT_DEB=klayout_0.30.12-1_amd64.deb
export EXTRA_CHECK_VERSION=wafer.space-run1
export CVC_VERSION=master
export DATA_ROOT=/content/design_data
export CHECK_VERSION=main
export LVS_ROOT=/root/extra_be_checks
export MPW_ROOT=/content/ws-mpw-be
export MPW=ws-run1
export SHUTTLE_ID=G801
if [[ -f $DATA_ROOT/project_env ]]; then
  cat $DATA_ROOT/project_env
  source $DATA_ROOT/project_env
fi
export LAYOUT_TOP=TTPG_tt_gf_wrapper_6_2
# export LAYOUT_TOP=2975_chip_top_0_6
export PROJECT=${LAYOUT_TOP}
export CHIP_ID=${LAYOUT_TOP%%_*}
export UPRJ_ROOT=${DATA_ROOT}/${CHIP_ID}
export WORK_ROOT=${DATA_ROOT}/${CHIP_ID}/work
export CHIP_ROOT=${MPW_ROOT}/layout/${CHIP_ID}
export LAYOUT_DATA=$

## 3. Create the cvcrc setup file and power file.

In [22]:
%%shell
cat /content/env
source /content/env
cat > ${WORK_ROOT}/cvcrc <<EOF
CVC_TOP = ${LAYOUT_TOP}
CVC_NETLIST = ${WORK_ROOT}/ext/${LAYOUT_TOP}.cdl.gz
CVC_MODEL_FILE = ${LVS_ROOT}/tech/${PDK}/cvc.models
CVC_POWER_FILE = ${WORK_ROOT}/cvc.power.${LAYOUT_TOP}
CVC_REPORT_FILE = ${WORK_ROOT}/cvc.log
EOF

VDD=5.0
VDD_33=("ISHI" "MOSB")
for cid in "${VDD_33[@]}"; do
  if [[ ${CHIP_ID} == ${cid} ]]; then
    VDD=3.3
  fi
done
VSS=0.0

rm -f ${WORK_ROOT}/cvc.power.${PROJECT}

mapfile -t add_pin_list < <(
    grep -v -e '^VDD' -e '^VSS' -e '_PAD' "${WORK_ROOT}/pin.list" |
    sed -E 's/\[[0-9]+\]/*/' |
    sort -u
)

# Use pin.list for CVC power settings when non-default pins exist
if ((${#add_pin_list[@]} > 0)); then

  for pin in "${add_pin_list[@]}"; do
    if [[ ${pin} == *VDD* ]] || [[ ${pin} == *vdd* ]]; then
      echo "${pin} power ${VDD}" >> ${WORK_ROOT}/cvc.power.${PROJECT}
    elif [[ ${pin} == *VSS* ]] || [[ ${pin} == *vss* ]]; then
      echo "${pin} power ${VSS}" >> ${WORK_ROOT}/cvc.power.${PROJECT}
    else
      cat >> ${WORK_ROOT}/cvc.power.${PROJECT} <<EOF
${pin} input min@${VSS} max@${VDD}
EOF
    fi
  done

else

# Default CVC power settings
  cat > ${WORK_ROOT}/cvc.power.${PROJECT} <<EOF
VDD power ${VDD}
VSS power ${VSS}
*_PAD* input min@VSS max@VDD
EOF

fi

if (( $(grep -c 'gf180mcu_fd_ip_sram__sram' ${CELL_LIST}) > 0 )); then
cat >> ${WORK_ROOT}/cvc.power.${PROJECT} <<EOF
*(gf180mcu_fd_ip_sram*)/CLK expectMin@VSS expectMax@VDD
*(gf180mcu_fd_ip_sram*)/CEN expectMin@VSS expectMax@VDD
*(gf180mcu_fd_ip_sram*)/A[*] expectMin@VSS expectMax@VDD
*(gf180mcu_fd_ip_sram*)/GWEN expectMin@VSS expectMax@VDD
*(gf180mcu_fd_ip_sram*)/WEN[*] expectMin@VSS expectMax@VDD
*(gf180mcu_fd_ip_sram*)/D[*] expectMin@VSS expectMax@VDD
*(gf180mcu_fd_ip_sram*)/Q[*] input min@VSS max@VDD
*(gf180mcu_fd_ip_sram*)/VDD expectSim@5.0
*(gf180mcu_fd_ip_sram*)/VSS expectSim@0.0
EOF
fi

ls -l ${WORK_ROOT}/cvc.power.${PROJECT}
cat ${WORK_ROOT}/cvc.power.${PROJECT}

export LOCAL_INSTALL=/content/local
export PATH=$PATH:$LOCAL_INSTALL/bin
export PDK_ROOT=/content/pdks
export PDK=gf180mcuD
export PDKPATH=$PDK_ROOT/$PDK
export PDK_COMMIT=f6eeac7dad085ffcc829ccfd721f7b4ce39edcf7
export MAGIC_VERSION=8.3.693
export NETGEN_VERSION=1.5.323
export KLAYOUT_DISTRO=Ubuntu-24
export KLAYOUT_DEB=klayout_0.30.12-1_amd64.deb
export EXTRA_CHECK_VERSION=wafer.space-run1
export CVC_VERSION=master
export DATA_ROOT=/content/design_data
export CHECK_VERSION=main
export LVS_ROOT=/root/extra_be_checks
export MPW_ROOT=/content/ws-mpw-be
export MPW=ws-run1
export SHUTTLE_ID=G801
if [[ -f $DATA_ROOT/project_env ]]; then
  cat $DATA_ROOT/project_env
  source $DATA_ROOT/project_env
fi
export LAYOUT_TOP=TTPG_tt_gf_wrapper_6_2
# export LAYOUT_TOP=2975_chip_top_0_6
export PROJECT=${LAYOUT_TOP}
export CHIP_ID=${LAYOUT_TOP%%_*}
export UPRJ_ROOT=${DATA_ROOT}/${CHIP_ID}
export WORK_ROOT=${DATA_ROOT}/${CHIP_ID}/work
export CHIP_ROOT=${MPW_ROOT}/layout/${CHIP_ID}
export LAYOUT_DATA=$

---
# ※ Run BE checks


In [ ]:
%%shell
cat /content/env
source /content/env
# run_be_checks includes magic/netgen soft connection and LVS, and CVC-RV
# ${LVS_ROOT}/run_be_checks ${WORK_ROOT}/lvs_config.json
CVC_NETLIST=${WORK_ROOT}/ext/${LAYOUT_TOP}.cdl.gz

if [[ -e ${CVC_NETLIST} ]]; then
  ${LVS_ROOT}/run_cvc --noextract ${WORK_ROOT}/lvs_config.json
else
  ${LVS_ROOT}/run_cvc ${WORK_ROOT}/lvs_config.json
fi
echo "Exit code = $?"
exit 0

python3 $PDK_ROOT/$PDK/libs.tech/klayout/tech/lvs/run_lvs.py --layout=$LAYOUT_DATA \
  --netlist=$CDL_FILE \
  --run_dir=$WORK_ROOT/klayout \
  --topcell=$LAYOUT_TOP \
  --run_mode=deep \
  --spice_comments \
  --no_simplify \
  --combine_devices \
  --top_lvl_pins

LVS_BASENAME=$( echo $LAYOUT_DATA | sed -e 's,.*/,,' -e 's,\..*,,' )
$LVS_ROOT/check_klayout_ports $WORK_ROOT/klayout/$LVS_BASENAME.lvsdb > $WORK_ROOT/klayout/port_check.log
if [[ -e $WORK_ROOT/klayout/port_check.log ]]; then
  if [[ -s $WORK_ROOT/klayout/port_check.log ]]; then
    echo "* Klayout ports do not match *"
    echo "layout     source"
    awk '{printf "%-10s %s\n", $1, $2}' $WORK_ROOT/klayout/port_check.log
  else
    echo "Klayout ports match"
  fi
fi

Streaming output truncated to the last 5000 lines.
Processing RW_gf180mcu_fd_sc_mcu7t5v0__and2_1
Processing RW_gf180mcu_fd_sc_mcu7t5v0__aoi22_1
Processing RW_gf180mcu_fd_sc_mcu7t5v0__xor2_2
Processing RW_gf180mcu_fd_sc_mcu7t5v0__aoi21_2
Processing RW_gf180mcu_fd_sc_mcu7t5v0__clkinv_2
Processing RW_gf180mcu_fd_sc_mcu7t5v0__tieh
Processing RW_gf180mcu_fd_sc_mcu7t5v0__inv_1
Processing RW_gf180mcu_fd_sc_mcu7t5v0__nand2_2
Processing RW_gf180mcu_fd_sc_mcu7t5v0__buf_4
Processing RW_gf180mcu_fd_sc_mcu7t5v0__nor2_4
Processing RW_gf180mcu_fd_sc_mcu7t5v0__aoi22_2
Processing RW_gf180mcu_fd_sc_mcu7t5v0__aoi221_4
Processing RW_gf180mcu_fd_sc_mcu7t5v0__buf_8
Processing RW_gf180mcu_fd_sc_mcu7t5v0__aoi22_4
Processing RW_gf180mcu_fd_sc_mcu7t5v0__oai21_2
Processing RW_gf180mcu_fd_sc_mcu7t5v0__buf_3
Processing RW_gf180mcu_fd_sc_mcu7t5v0__clkbuf_3
Processing RW_gf180mcu_fd_sc_mcu7t5v0__nor3_2
Processing RW_gf180mcu_fd_sc_mcu7t5v0__tiel
Processing RW_gf180mcu_fd_sc_mcu7t5v0__nand4_2
Processing RW_gf180mcu_f

# View Extract and CVC logs

In [ ]:
import os
import subprocess
from google.colab import files

result = subprocess.run(
    "source /content/env && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        k, v = line.split("=", 1)
        os.environ[k] = v

work_root = os.environ["WORK_ROOT"]
files.view(f"{work_root}/ext.log")
files.view(f"{work_root}/cvc.log")


# View CVC error report

(1) Uncompress cvc.error.gz file

In [ ]:
%%shell
source /content/env
gzip -df ${WORK_ROOT}/cvc.error.gz

(2) View cvc.error file

In [ ]:
import os
import subprocess
from google.colab import files

result = subprocess.run(
    "source /content/env && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        k, v = line.split("=", 1)
        os.environ[k] = v

work_root = os.environ["WORK_ROOT"]
files.view(f"{work_root}/cvc.error")